most similar objects with cosine similarity

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from PIL import Image

EMBED_PATH = Path('multimodal_embeddings_gemini_2.pt')
IMAGES_DIR = Path('images')
QUERY_OBJECT_ID = '4002979192'
TOP_K = 10

In [ ]:
db = torch.load(EMBED_PATH, map_location='cpu')
embeddings = db['embeddings'].float()
rows = db['rows']

# normalize vectors so dot product -> cosine similarity
embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)

object_ids = [row['object_id'] for row in rows]
if QUERY_OBJECT_ID not in object_ids:
    raise ValueError(f'Unknown object_id: {QUERY_OBJECT_ID}')

query_idx = object_ids.index(QUERY_OBJECT_ID)

print('embeddings shape:', tuple(embeddings.shape))
print('objects:', len(rows))
print('query object_id:', QUERY_OBJECT_ID)

In [ ]:
query_row = rows[query_idx]
query_images = sorted((IMAGES_DIR / QUERY_OBJECT_ID).glob('*.jpg'))

print('QUERY OBJECT')
print('object_id:', QUERY_OBJECT_ID)
print('text_prompt:\n', query_row['text_prompt'])
print('n_images:', len(query_images))

for p in query_images:
    img = Image.open(p).convert('RGB')
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(p.name, fontsize=9)
    plt.axis('off')
    plt.show()

In [ ]:
q = embeddings[query_idx]
sims = embeddings @ q # cosine distance
sims[query_idx] = -1  # remove the query itself

top_scores, top_idxs = torch.topk(sims, k=TOP_K)

print('TOP 3 MOST SIMILAR OBJECTS')
for i in range(TOP_K):
    idx = top_idxs[i].item()
    score = top_scores[i].item()
    print(f"#{i+1} | object_id={rows[idx]['object_id']} | cosine={score:.4f}")

In [ ]:
for i, idx in enumerate(indices[0][:TOP_K]):
    score = cosine_similarities[0][idx].item()
    row = rows[idx]
    
    image_path = IMAGES_DIR / f"{row['id']}.jpg"
    
    print(f"#{i+1} | score={score:.4f} | object_id={row['object_id']}")
    print(row['short_description'])
    print(image_path)
    print('-' * 80)


In [ ]:
# 1. Fetch reference (query) object details
query_image_paths = sorted((IMAGES_DIR / QUERY_OBJECT_ID).glob('*.jpg'))

# 2. First pass: find the maximum number of images across the reference AND all TOP_K results
max_images = len(query_image_paths)
for i in range(TOP_K):
    idx = top_idxs[i].item()
    object_id = rows[idx]['object_id']
    n_images = len(list((IMAGES_DIR / object_id).glob('*.jpg')))
    max_images = max(max_images, n_images)

# Columns = 1 (for text metadata) + max_images (for the right-side images)
cols = max_images + 1
total_rows = TOP_K + 1

# Make the text column slightly wider (1.5) than the image columns (1.0) so it fits neatly
width_ratios = [1.5] + [1] * max_images

# Smaller overall figsize to strictly pack spacing
fig, axes = plt.subplots(
    total_rows, 
    cols, 
    figsize=(2 * max_images + 3, 1.8 * total_rows), 
    squeeze=False,
    gridspec_kw={'width_ratios': width_ratios}
)

fig.suptitle("Nearest Neighbors Retrieval", fontsize=14, fontweight='bold', y=1.0)

# Helper function to plot a single row cleanly
def plot_row(row_idx, obj_id, title, score_text, img_paths, is_query=False):
    text_ax = axes[row_idx, 0]
    text_ax.axis('off')
    
    if is_query:
        title_str = f"⭐ REFERENCE ⭐"
        text_weight = 'bold'
    else:
        title_str = f"{title}"
        text_weight = 'normal'
        
    info_text = (
        f"{title_str}\n"
        f"ID: {obj_id}\n"
        f"{score_text}"
        f"Images: {len(img_paths)}"
    )
    
    text_ax.text(0.0, 0.5, info_text, va='center', ha='left', fontsize=11, weight=text_weight)
    
    for j in range(1, cols):
        img_ax = axes[row_idx, j]
        img_idx = j - 1
        
        if img_idx < len(img_paths):
            p = img_paths[img_idx]
            img = Image.open(p).convert('RGB')
            img_ax.imshow(img)
            # Add small padding to titles to bring them closer to the image
            img_ax.set_title(p.name, fontsize=8, fontweight=text_weight, pad=2)
            
        img_ax.axis('off')

# ------------------ 0. PLOT REFERENCE OBJECT ------------------
plot_row(
    row_idx=0, 
    obj_id=QUERY_OBJECT_ID, 
    title="REFERENCE", 
    score_text="Query Image\n", 
    img_paths=query_image_paths,
    is_query=True
)

# ----------------- 1. PLOT TOP K RESULTS -----------------
for i in range(TOP_K):
    idx = top_idxs[i].item()
    score = top_scores[i].item()
    row = rows[idx]
    object_id = row['object_id']
    
    image_paths = sorted((IMAGES_DIR / object_id).glob('*.jpg'))
    
    plot_row(
        row_idx=i + 1, 
        obj_id=object_id, 
        title=f"#{i+1} SIMILAR", 
        score_text=f"Similarity: {score:.4f}\n", 
        img_paths=image_paths,
        is_query=False
    )

# Force very tight padding between the images horizontally and vertically
plt.tight_layout(pad=0.5, h_pad=0.1, w_pad=0.1)
plt.show()
